# Guided exercise: clean and total Kalpa's orders

**Week 0, foundations guide, Chapter 1: Python.** Seven steps that fit the chapter's ninety
minutes of hands-on, build mini project 1, and end on the three cells you copy into your
`w00-diagnostic-python` repository.

> **Meera Raghavan, CEO of Kalpa Retail:** "Revenue fell from Q1 to Q2, where did it go?"

Her question needs revenue by tier, and revenue by tier needs numbers. The order rows arrive the way
a CSV delivers them, with every field as text, so an amount such as `'1,200'` has to be converted
before it can be added, and an amount such as `'n/a'` cannot be converted at all. Anand asked in Q9
of the diagnostic for every bad value to be skipped and counted, whatever it looks like, and the
loop you build here does that.

The ten rows are constructed for practice. Their amounts are examples the guide itself uses, such
as `'1,200'`, `'950'`, `'n/a'` and `'800'`, and none of them is Kalpa's data.

**How the placeholders work.** Every `__TODO1__`, `__TODO2__` and so on is yours to replace. A
lettered placeholder lists four options in the comment above it, and you replace it with the code
of the letter you pick. A prediction asks for the value you expect, written before you run the
cell that produces it. The trace table and the closing sentence are yours to write. Run as
shipped, the notebook stops with a `NameError` at the first placeholder, which is where you
start, and the checks under each step say whether you were right.

## Step 1. Redraw the board before any code (10 minutes)

Close the chapter and redraw its picture from memory on paper: the board of names and objects
that Ned Batchelder built his PyCon 2015 talk around, with the names on the left and the objects
on the right. Draw what `b = rows` does to the board, then what `b[1] = ...` does, and then what
`test = rows.copy()` would have done instead. Give it five minutes before you run the next cell.

The next cell finds the programme's helper, `scripts/c2kit.py`, by walking up from this folder,
and draws the same board, so you can compare the two drawings arrow by arrow.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

kit.side_by_side(
    kit.sequence(["rows", "b", "one list"],
                 [("rows", "one list", "rows = [ten tuples]"),
                  ("b", "one list", "b = rows"),
                  ("b", "one list", "b[1] = ... changes the list"),
                  ("rows", "one list", "rows[1] shows the change")],
                 title="b = rows: two names, one list", show=False),
    kit.sequence(["rows", "test", "list one", "list two"],
                 [("rows", "list one", "rows = [ten tuples]"),
                  ("test", "list two", "test = rows.copy()"),
                  ("test", "list two", "test[1] = ... changes list two"),
                  ("rows", "list one", "rows[1] is unchanged")],
                 title="test = rows.copy(): two names, two lists", show=False),
)

An arrow on the board is a name bound to an object. `b = rows` adds an arrow and builds nothing,
so a change made through `b` lands in the one list both names point at, and `rows.copy()` is what
builds a second list. If your drawing had `b = rows` copying the list, step 2 is where that model
meets Kalpa's rows. The map below places the exercise on Meera's thread, Figure 1 of the guide:
this chapter carries the rows from raw text to totals by tier.

In [ ]:
kit.flow(["raw rows\nid, tier, amount, status", "clean\n'1,200' -> 1200, 'n/a' rejected",
          "aggregate\nrevenue by tier", "reason\nwhich tier moved", "label\nrefund or delivery",
          "say it\none sentence"],
         kinds=["known", "lit", "lit", "unknown", "unknown", "unknown"],
         title="Meera's question from raw rows to the sentence she hears: this exercise runs the two dark steps")

## Step 2. Cell one: the ten rows, and two names for one list (10 minutes)

Cell one holds the rows as tuples of `(order_id, tier, amount_text, status)`, with every field
as text, the way a CSV would deliver them. Run it as it is.

In [ ]:
# Mini project cell one: ten order rows, every field text, the way a CSV delivers them
rows = [
    ("K-101", "Plus", "1,200", "paid"),
    ("K-102", "Basic", "950", "paid"),
    ("K-103", "Plus", "n/a", "paid"),
    ("K-104", "Basic", "300", "paid"),
    ("K-105", "Plus", "800", "cancelled"),
    ("K-106", "Basic", "900", "paid"),
    ("K-107", "Plus", "", "paid"),
    ("K-108", "Plus", "1,200", "paid"),
    ("K-109", "Basic", "Rs 900", "paid"),
    ("K-110", "Plus", "950", "paid"),
]
print(len(rows), "rows; the first is", rows[0])

The self-check at the end asks you to change one amount to `'0'` and watch the row survive. A
quick way to try that without spoiling the ten rows seems to be a second name for them. Before
you run the next cell, write in `predicted_seen` the text you expect `rows[1][2]` to show once
K-102's amount has been changed through `b`.

In [ ]:
# TODO 1. Your prediction, written before you run this cell: the text rows[1][2] will show.
predicted_seen = __TODO1__

b = rows                                      # a second name for the list
b[1] = ("K-102", "Basic", "0", "paid")        # the self-check's change, made through b
seen_through_rows = rows[1][2]                # what the first name shows now
rows[1] = ("K-102", "Basic", "950", "paid")   # put the original row back
print("rows[1][2] showed", repr(seen_through_rows), "and b is rows:", b is rows)

In [ ]:
kit.check("your prediction matched what rows[1][2] showed", predicted_seen == seen_through_rows,
          f"you predicted {predicted_seen!r}, rows showed {seen_through_rows!r}")
kit.check("b and rows are one object, which is what the test a is b answers", b is rows)
kit.check("the original K-102 row is back in place", rows[1] == ("K-102", "Basic", "950", "paid"),
          str(rows[1]))

Before any conversion runs, read the ten amounts and decide which of them no conversion can turn
into a number. Write their order ids in `planted_bad`, in row order. These are the rows the loop
should reject, and step 3 compares its rejects with your list.

In [ ]:
# TODO 2. The order ids of the rows whose amount cannot be read as a number, in row order.
planted_bad = __TODO2__
print("you expect", len(planted_bad), "rows to be rejected:", planted_bad)

## Step 3. Cell two: one conversion per value, one try per row (20 minutes)

The chapter's rule is that the conversion lives in one function, called on every row, that
either returns a number or raises. The loop then puts one `try` around that single call, so a bad
value costs one row and the loop carries on, which is the shape Anand's request in Q9 needs.
Figure 7 of the chapter draws it.

In [ ]:
kit.flow(["for row in rows\none row at a time", "try: convert\n'1,200' -> 1200.0",
          "ValueError?\n'n/a' raises", "keep the reject\nrejected.append(row)",
          "next row\nthe loop carries on"], lit=3,
         title="One try per value: a bad amount costs one row, and the loop carries on")

**Predict before you run.** Write both predictions in the next cell before you run cell two. For
each of the five texts, write the number `to_amount` will return, or the word `"ValueError"`
where it will raise. Then write the Plus total the loop will reach for paid orders.

In [ ]:
texts = ["1,200", "n/a", "", "Rs 900", "0"]

# TODO 3. What to_amount returns for each text above: a number, or "ValueError" where it raises.
predicted_returns = __TODO3__

# TODO 4. The Plus total for paid orders, as a number.
predicted_plus = __TODO4__
print("predictions recorded:", predicted_returns, predicted_plus)

Cell two holds three lettered placeholders. Fill all three before you run it, because the loop
reaches them in a different order from the one they are numbered in.

In [ ]:
# Mini project cell two: one conversion, and a loop with one try per row
def to_amount(text):
    """The amount written in text, as a number; raises ValueError when the text is not a number."""
    # TODO 5. Which expression turns '1,200' into 1200.0 and raises on a value that is not a number?
    #   a) float(text)
    #   b) float(text.strip(","))
    #   c) float(text.replace(",", ""))
    #   d) float(text.replace(",", "") or 0)
    return __TODO5__


def clean_and_total(rows):
    """Totals by tier for paid orders, the clean paid rows, and the rows whose amount would not convert."""
    totals, clean, rejected = {}, [], []
    for row in rows:
        order_id, tier, amount_text, status = row
        try:
            amount = to_amount(amount_text)
        except ValueError:
            # TODO 6. What should happen to a row whose amount cannot be read?
            #   a) rejected.append(row); continue
            #   b) rejected.append(row)
            #   c) rejected.append(amount_text); continue
            #   d) break
            __TODO6__
        if status == "paid":
            # TODO 7. Which line adds this order's amount to its tier's running total?
            #   a) totals[tier] = amount
            #   b) totals[tier] = totals[tier] + amount
            #   c) totals[tier] = totals.get(tier, amount) + amount
            #   d) totals[tier] = totals.get(tier, 0) + amount
            __TODO7__
            clean.append((order_id, tier, amount))
    return totals, clean, rejected


totals, clean, rejected = clean_and_total(rows)
print(len(rows), "rows in:", len(clean), "clean and paid,", len(rejected), "rejected")

In [ ]:
actual_returns = []
for text in texts:
    try:
        actual_returns.append(to_amount(text))
    except ValueError:
        actual_returns.append("ValueError")
kit.table(["text", "you predicted", "to_amount gave"],
          [(repr(t), repr(p), repr(a)) for t, p, a in zip(texts, predicted_returns, actual_returns)],
          caption="Five texts through to_amount")

unpaid = sum(1 for r in rows if r[3] != "paid")
predicted = ["ValueError" if p is ValueError else p for p in predicted_returns]
kit.check("your five predictions for to_amount matched", predicted == actual_returns,
          f"predicted {predicted}, got {actual_returns}")
kit.check("your Plus total matched the loop's", predicted_plus == totals.get("Plus"),
          f"predicted {predicted_plus}, the loop reached {totals.get('Plus')}")
kit.check("every row is clean, rejected or unpaid, and none is lost or counted twice",
          len(clean) + len(rejected) + unpaid == len(rows),
          f"{len(clean)} clean + {len(rejected)} rejected + {unpaid} unpaid, from {len(rows)} rows")
kit.check("the loop rejected exactly the rows you listed in planted_bad",
          rejected == [r for r in rows if r[0] in planted_bad], f"rejected {rejected}")

## Step 4. Trace the loop by hand (12 minutes)

The chapter's method for a loop is a table: one row per iteration and one column per variable,
with the comparison written out where it matters. Trace the first six rows through
`clean_and_total` on paper, then type your table into the next cell. The first row is filled in
for you.

In [ ]:
# TODO 8. One tuple per row for K-101 to K-106, in loop order:
# (order_id, what happened, Plus total after the row, Basic total after the row),
# where what happened is "added", "rejected" or "skipped".
my_trace = [
    ("K-101", "added", 1200, 0),
    __TODO8__
]
print(len(my_trace), "rows traced")

The next cell builds the real trace without changing your function. It runs `clean_and_total` on
the first row, then on the first two, and so on up to six, because the totals after the first k
rows are exactly the running totals after row k, and whichever list grew between two runs says
what happened to the row that was added.

In [ ]:
real_trace = []
before = clean_and_total([])
for k in range(1, 7):
    after = clean_and_total(rows[:k])
    if len(after[2]) > len(before[2]):
        happened = "rejected"
    elif len(after[1]) > len(before[1]):
        happened = "added"
    else:
        happened = "skipped"
    real_trace.append((rows[k - 1][0], happened, after[0].get("Plus", 0), after[0].get("Basic", 0)))
    before = after

kit.table(["order_id", "what happened", "Plus after", "Basic after", "your row matched"],
          [(oid, what, kit.rupees(plus), kit.rupees(basic),
            "yes" if k < len(my_trace) and tuple(my_trace[k]) == real_trace[k] else "no")
           for k, (oid, what, plus, basic) in enumerate(real_trace)],
          caption="The real trace of the first six rows, against yours")
kit.line([t[0] for t in real_trace],
         [("Plus", [t[2] for t in real_trace], "lit"), ("Basic", [t[3] for t in real_trace], "")],
         fmt=kit.rupees, title="Running totals after each of the first six rows")

In [ ]:
matched = sum(1 for mine, real in zip(my_trace, real_trace) if tuple(mine) == real)
kit.check("your trace matches the loop row for row",
          [tuple(t) for t in my_trace] == real_trace, f"{matched} of {len(real_trace)} rows match")
kit.check("a rejected or skipped row leaves both totals where they were",
          all(real_trace[k][2:] == real_trace[k - 1][2:]
              for k in range(1, len(real_trace)) if real_trace[k][1] != "added"))

## Step 5. Cell three: print the answer and prove it adds up (8 minutes)

An assert is only worth writing if it can fail. The mini project asks for one that compares the
sum of the totals with the sum of the clean amounts, two numbers the loop builds separately, so a
total that lost an amount or counted one twice shows up as a mismatch.

In [ ]:
# Mini project cell three: the totals, the rejects, and an assert that the two sums agree
def totals_reconcile(totals, clean):
    # TODO 9. Which comparison proves the totals lost no clean amount and counted none twice?
    #   a) sum(totals.values()) == sum(totals.values())
    #   b) sum(totals.values()) == sum(a for _, _, a in clean)
    #   c) sorted(totals) == sorted({tier for _, tier, _ in clean})
    #   d) sum(totals.values()) > 0
    return __TODO9__


print("totals by tier:", totals)
print("rejected rows:", rejected)
assert totals_reconcile(totals, clean), "the totals and the clean amounts disagree"
print("the sum of the totals equals the sum of the clean amounts:", sum(totals.values()))

In [ ]:
kit.bridge(("rows in", len(rows)), [("rejected", -len(rejected)), ("not paid", -unpaid)],
           end_label="clean and paid", fmt=lambda v: f"{v:.0f}",
           title="Every row accounted for: rows in, less the rejects and the unpaid")

broken = dict(totals)      # a copy, so breaking it leaves totals as it is
broken["Plus"] = 0.0
kit.check("your assert holds on the real totals", totals_reconcile(totals, clean))
kit.check("your assert fails when a total is broken on purpose, so it can catch a real mistake",
          not totals_reconcile(broken, clean))
kit.check("totals was left alone by the broken copy", totals.get("Plus", 0) > 0, str(totals))

## Step 6. Break it on purpose: the zero a truthiness test drops (17 minutes)

The self-check's last line says that changing one amount to `'0'` must keep the row, because a
truthiness test would drop it. Step 2 showed that `b = rows` cannot hold that experiment, since
the change lands in `rows` itself, so start by making a second list to change. If the checks
under the next cell say `rows` changed, run cell one again to restore the ten rows before you go
on.

In [ ]:
# TODO 10. Which expression gives you a second list to change, leaving rows as it is?
#   a) rows
#   b) rows[1:]
#   c) rows[:]
#   d) rows.copy
zero_rows = __TODO10__
zero_rows[1] = ("K-102", "Basic", "0", "paid")
print("zero_rows[1] is", zero_rows[1], "and rows[1] is", rows[1])

In [ ]:
kit.check("zero_rows is a second list", zero_rows is not rows)
kit.check("zero_rows holds all ten rows, with K-102 at '0'",
          len(zero_rows) == len(rows) and zero_rows[1][2] == "0", f"{len(zero_rows)} rows")
kit.check("rows still holds K-102 at '950'", rows[1][2] == "950", repr(rows[1][2]))

A colleague reviewing cell two calls the `try` block long-winded and suggests a shortcut: convert
with a function that hands back `None` for a bad value, and keep a row whenever it has an amount,
which is the one-word test `if amount:`. It reads well, and on the original ten rows it gives
exactly the answer cell two gives. Make the colleague's change on purpose by replacing
`__TODO12__` with the one word `amount`, and before you run the cell, predict how many rows the
shortcut will reject from `zero_rows`.

In [ ]:
# TODO 11. Your prediction, written before you run this cell: how many rows the shortcut rejects.
predicted_shortcut_rejects = __TODO11__


def to_amount_or_none(text):
    """The colleague's converter: a number, or None when the text is not a number."""
    try:
        return to_amount(text)
    except ValueError:
        return None


def shortcut_clean(rows):
    totals, clean, rejected = {}, [], []
    for row in rows:
        order_id, tier, amount_text, status = row
        amount = to_amount_or_none(amount_text)
        # TODO 12. The colleague's test, typed on purpose: the one word amount.
        if __TODO12__:
            if status == "paid":
                totals[tier] = totals.get(tier, 0) + amount
                clean.append((order_id, tier, amount))
        else:
            rejected.append(row)
    return totals, clean, rejected


short_totals, short_clean, short_rejected = shortcut_clean(zero_rows)
print("the shortcut rejected", len(short_rejected), "rows:", [r[0] for r in short_rejected])

**The plausible wrong answer.** The next cell sets the shortcut beside cell two on the same
`zero_rows`, where K-102's amount is `'0'`.

In [ ]:
def basic_orders(clean):
    return sum(1 for _, tier, _ in clean if tier == "Basic")


good_totals, good_clean, good_rejected = clean_and_total(zero_rows)
kit.table(["on zero_rows", "cell two", "the colleague's shortcut"],
          [("rows rejected", len(good_rejected), len(short_rejected)),
           ("Basic orders kept", basic_orders(good_clean), basic_orders(short_clean)),
           ("Basic revenue", kit.rupees(good_totals["Basic"]), kit.rupees(short_totals["Basic"])),
           ("Basic average order", kit.rupees(good_totals["Basic"] / basic_orders(good_clean)),
            kit.rupees(short_totals["Basic"] / basic_orders(short_clean)))],
          caption="The ten rows with K-102 at '0', cleaned two ways")
kit.columns(["rows rejected", "Basic orders kept"],
            [("cell two", [len(good_rejected), basic_orders(good_clean)]),
             ("the shortcut", [len(short_rejected), basic_orders(short_clean)])],
            title="Cell two against the shortcut on zero_rows")

**Why it is wrong.** K-102 is a real Basic order whose amount is zero, and `if amount:` treats
0.0 exactly as it treats `None`, because both are false in an `if` test, which was Q6 of the
diagnostic. The shortcut files a good order with the unreadable ones. Anand's count of bad values
rises by one and sends someone to repair a row that was never broken, and the Basic tier loses an
order, which lifts its average order value by half. Order value by tier is what Q29 of the
diagnostic turned on, so this is the number that would tell Meera the wrong story about what
Basic customers spend. Cell three's assert cannot see any of it, because a zero adds nothing to
either sum. The check that catches it is the chapter's tell: a reject has to be a value that
fails to convert, so count the rejects whose amount converts.

In [ ]:
converts = [r[0] for r in short_rejected if to_amount_or_none(r[2]) is not None]
kit.check("your prediction matched the shortcut's reject count",
          predicted_shortcut_rejects == len(short_rejected),
          f"predicted {predicted_shortcut_rejects}, the shortcut rejected {len(short_rejected)}")
kit.check("caught: the shortcut rejects a row whose amount converts", len(converts) > 0,
          f"rejected although it converts: {converts}")
kit.check("on the original ten rows the shortcut and cell two agree, which is why it looked safe",
          shortcut_clean(rows) == clean_and_total(rows))

**The fix.** Ask the question you mean. The colleague meant "is there an amount", and `None` is
the only value that says there is none. Pick the test that keeps a genuine zero and still rejects
the three rows that do not convert.

In [ ]:
def fixed_clean(rows):
    totals, clean, rejected = {}, [], []
    for row in rows:
        order_id, tier, amount_text, status = row
        amount = to_amount_or_none(amount_text)
        # TODO 13. Which test keeps a genuine zero and still rejects a value that is not a number?
        #   a) amount
        #   b) amount is not None
        #   c) amount is not None and amount != 0
        #   d) amount != 0
        if __TODO13__:
            if status == "paid":
                totals[tier] = totals.get(tier, 0) + amount
                clean.append((order_id, tier, amount))
        else:
            rejected.append(row)
    return totals, clean, rejected


fixed_totals, fixed_rows, fixed_rejected = fixed_clean(zero_rows)
print("the fixed shortcut rejected", len(fixed_rejected), "rows and kept",
      basic_orders(fixed_rows), "Basic orders")

In [ ]:
kit.check("the fix keeps K-102 as a Basic order of Rs 0", ("K-102", "Basic", 0.0) in fixed_rows)
kit.check("the fix rejects exactly the rows you planted",
          [r[0] for r in fixed_rejected] == list(planted_bad), str([r[0] for r in fixed_rejected]))
kit.check("the fixed shortcut and cell two now agree on zero_rows",
          fixed_clean(zero_rows) == clean_and_total(zero_rows))

What changed: the rejects are back to the three planted rows, and Basic keeps all three of its
orders at an average of Rs 400, where the shortcut showed two at Rs 600. The fixed shortcut now
agrees with cell two, because both treat a reject as a value that failed to convert.

## Step 7. The self-check, the sentence for Meera, and the copy (13 minutes)

The guide's self-check has three lines, and each one is a check below, beside a fourth that holds
these totals for Chapter 6. The first line holds only when this run started from a fresh kernel,
so before you trust the result, restart the kernel and run every cell from the top. The check
counts cells, so delete any cell you added before that final run.

In [ ]:
cells_run = len(In) - 1   # In holds every cell this kernel has run, after an empty first entry
kit.check("the notebook ran top to bottom once from a fresh kernel, with no error on the way",
          cells_run == 23,
          f"this kernel has run {cells_run} cells, and a clean run reaches this one as cell 23")
kit.check("the rejected list holds exactly the bad rows planted: K-103, K-107 and K-109",
          rejected == [r for r in rows if r[0] in ("K-103", "K-107", "K-109")], str(rejected))
kit.check("changing K-102's amount to '0' keeps the row",
          "K-102" in [c[0] for c in clean_and_total(zero_rows)[1]])
kit.check("the totals are Plus Rs 3,350 and Basic Rs 2,150, the figures Chapter 6 must reproduce",
          totals == {"Plus": 3350.0, "Basic": 2150.0}, str(totals))
kit.table(["tier", "paid revenue"], [(t, kit.rupees(v)) for t, v in totals.items()],
          caption="Mini project 1's answer")

Close the thread where it opened. Write the one sentence you would now say to Meera about these
rows, and build it from the variables so the numbers cannot drift from the answer: name both tier
totals with `kit.rupees` and say how many amounts were set aside.

In [ ]:
# TODO 14. Your sentence for Meera, as an f-string built from totals and rejected.
say_back = __TODO14__
print(say_back)

In [ ]:
kit.check("your sentence carries both tier totals and the number of rows set aside",
          kit.rupees(totals["Plus"]) in say_back and kit.rupees(totals["Basic"]) in say_back
          and f" {len(rejected)} " in f" {say_back} ")

In [ ]:
kit.check_summary()

## Copy it into your repository

Create `w00-diagnostic-python` the way Chapter 8 shows: public, with a README and the Python
`.gitignore`. Add a notebook named `notebook.ipynb` and copy three cells into it, in order: the
cell that starts `# Mini project cell one`, the one that starts `# Mini project cell two`, and
the one that starts `# Mini project cell three`, deleting the lettered comment lines as you go.
They use nothing from the helper, so they run anywhere. Restart the kernel and run all, then run
the self-check by hand: the rejected list names exactly the rows you planted, and changing one
amount to `'0'` keeps that row. Put your sentence for Meera in the README, and post the
repository link in the Week 0 thread on Discussions.

Chapter 6's guided exercise saves these same ten rows as `orders.csv`, answers the question in
six lines of pandas, and holds itself to these totals to the rupee.